# Лабораторная работа № 2. Сервер статистики


In [ ]:
from xmlrpc.server import SimpleXMLRPCServer, SimpleXMLRPCRequestHandler
from pathlib import Path
from datetime import datetime
import csv


LOG_PATH = Path('log.csv')
MAX_RECORDS = 10
FIELDS = ['timestamp', 'event_type']


class RequestHandler(SimpleXMLRPCRequestHandler):
    rpc_paths = ('/RPC2',)


def read_events(path):
    if not path.exists():
        return []

    with path.open(encoding='utf-8-sig', newline='') as file:
        rows = list(csv.reader(file))

    if not rows:
        return []

    if rows[0] == FIELDS:
        return [dict(zip(FIELDS, row)) for row in rows[1:] if len(row) == 2]

    return [
        {'timestamp': row[1], 'event_type': row[0]}
        for row in rows
        if len(row) == 2
    ]


def new_log():
    with LOG_PATH.open('w', encoding='utf-8', newline='') as file:
        csv.writer(file).writerow(FIELDS)


def rotate_log():
    stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    archive = LOG_PATH.with_name(f'{LOG_PATH.stem}_{stamp}.csv')

    number = 1

    while archive.exists():
        archive = LOG_PATH.with_name(f'{LOG_PATH.stem}_{stamp}_{number}.csv')
        number += 1

    LOG_PATH.rename(archive)
    new_log()

    return archive.name


def archive_time(path):
    prefix = f'{LOG_PATH.stem}_'

    if not path.stem.startswith(prefix):
        return None

    timestamp = path.stem[len(prefix):]
    parts = timestamp.split('_')

    if len(parts) < 2:
        return None

    timestamp = f'{parts[0]}_{parts[1]}'

    try:
        return datetime.strptime(timestamp, '%Y%m%d_%H%M%S')
    except ValueError:
        return None


def archive_paths_with_time():
    archives = []

    for path in LOG_PATH.parent.glob(f'{LOG_PATH.stem}_*.csv'):
        timestamp = archive_time(path)

        if timestamp is not None:
            archives.append((path, timestamp))

    return sorted(archives, key=lambda item: (item[1], item[0].name))


def select_archive_paths(lower=None, upper=None):
    archives = archive_paths_with_time()

    if lower is None and upper is None:
        return [path for path, _ in archives]

    selected = []

    for index, (path, end_time) in enumerate(archives):
        start_time = archives[index - 1][1] if index > 0 else None

        if lower is not None and end_time < lower:
            continue

        if upper is not None and start_time is not None and start_time > upper:
            continue

        selected.append(path)

        if selected and upper is not None:
            last_index = next(
            i for i, (path, _) in enumerate(archives)
            if path == selected[-1]
        )

        if last_index + 1 < len(archives):
            next_path = archives[last_index + 1][0]

            if any(
                (lower is None or datetime.fromisoformat(e['timestamp']) >= lower)
                and datetime.fromisoformat(e['timestamp']) <= upper
                for e in read_events(next_path)
            ):
                selected.append(next_path)
                
    return selected


if not isinstance(MAX_RECORDS, int) or MAX_RECORDS < 1:
    raise ValueError('MAX_RECORDS должен быть положительным целым числом')


if not LOG_PATH.exists() or LOG_PATH.stat().st_size == 0:
    new_log()
else:
    with LOG_PATH.open(encoding='utf-8-sig', newline='') as file:
        header = next(csv.reader(file), [])

    if header != FIELDS:
        rotate_log()


def add_log(event_type, event_time=''):
    timestamp = event_time or datetime.now().isoformat(timespec='microseconds')

    datetime.fromisoformat(timestamp)

    if len(read_events(LOG_PATH)) >= MAX_RECORDS:
        rotate_log()

    with LOG_PATH.open('a', encoding='utf-8', newline='') as file:
        csv.writer(file).writerow([timestamp, event_type])

    return True


def get_log(event_type='', time_from='', time_to='', include_archives=True):
    lower = datetime.fromisoformat(time_from) if time_from else None
    upper = datetime.fromisoformat(time_to) if time_to else None

    if lower and upper and lower > upper:
        raise ValueError('Начало интервала позже конца')

    paths = select_archive_paths(lower, upper) if include_archives else []
    paths.append(LOG_PATH)

    events = []

    for path in paths:
        for event in read_events(path):
            moment = datetime.fromisoformat(event['timestamp'])

            if event_type and event['event_type'] != event_type:
                continue

            if lower and moment < lower:
                continue

            if upper and moment > upper:
                continue

            events.append(event)

    return sorted(
        events,
        key=lambda event: datetime.fromisoformat(event['timestamp'])
    )



def get_status():
    archives = [path.name for path, _ in archive_paths_with_time()]

    return {
        'max_records': MAX_RECORDS,
        'current_records': len(read_events(LOG_PATH)),
        'archives': archives
    }


server = SimpleXMLRPCServer(
    ('localhost', 8018),
    requestHandler=RequestHandler,
    logRequests=False,
    allow_none=True
)

server.register_function(add_log, 'add_log')
server.register_function(get_log, 'get_log')
server.register_function(get_status, 'get_status')

print(f'Сервер статистики: http://localhost:8018/RPC2; лимит: {MAX_RECORDS}')

server.serve_forever()